In [1]:
# 1. 確認所需套件的版本
import torch
print("PyTorch 的版本為: {}".format(torch.__version__))

import transformers
print("Hugging Face Transformers 的版本為: {}".format(transformers.__version__))

import datasets
print("Hugging Face Datasets 的版本為: {}".format(datasets.__version__))

## dataset如果有問題，記得要重安裝 pip install --upgrade pandas pandasai

PyTorch 的版本為: 2.4.1+cu121


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face Transformers 的版本為: 4.28.0
Hugging Face Datasets 的版本為: 3.2.0


In [2]:
import wandb

# 手動設置 API 金鑰
wandb.login(key="你的_wandb_api_key")


Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: soaring0616 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [16]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="output2.json")
train_dataset = dataset["train"]

print(train_dataset[0:11])  # 查看前十筆數據


{'question': ['中醫治療哪些病最有效？', '為什麼要看中醫？', '什麼病可以看中醫？', '中醫治療有哪些？', '中醫要持續看嗎？', '皮膚病看中醫有效嗎？', '中醫要看幾次？', '看中醫要看多久？', '同一天可以看西醫跟中醫嗎？', '中醫最多開幾天藥？', '吃中藥隔多久才能吃西藥？'], 'label': ['療效', '其他', '治療範疇', '治療範疇', '其他', '療效', '診斷方法', '其他', '其他', '其他', '其他']}


#### 訓練資料的分布

| label | 安全性 | 其他 | 治療方法 | 治療範疇 | 注意事項 | 診斷方法 | 適用人群 | 療效 | 總和 |
| -------- | -------- | -------- | -------- | -------- | -------- | -------- | -------- | -------- | -------- | 
| 該類的 question 數量 | 23 | 62 | 161 | 67 | 22 | 46 | 13 | 48 | 442 |

In [5]:
from transformers import AutoTokenizer

model_name = "bert-base-chinese"  # 你可以選擇適合的模型，如 Llama-2、ChatGLM
tokenizer = AutoTokenizer.from_pretrained(model_name)

def Instruction_preprocess_function(examples):
    inputs = [f"指令：{instr} \n輸入：{inp}" for instr, inp in zip(examples["instruction"], examples["input"])]
    targets = examples["output"]
    
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def preprocess_function(examples):
    inputs = examples["question"]
    targets = examples["label"]
    
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = train_dataset.map(preprocess_function, batched=True)
tokenized_datasets


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 442/442 [00:00<00:00, 1957.41 examples/s]


Dataset({
    features: ['question', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 442
})

In [8]:
# # 轉換資料格式：將 'input' 和 'instruction' 組合
# inputs = data['input']
# instructions = data['instruction']

inputs = dataset["train"]['question']
instructions = dataset["train"]['label']
instruction_list = list(set(instructions))  

In [9]:
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
import torch
from sklearn.model_selection import train_test_split

# 初始化tokenizer和模型
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=len(instruction_list))  # 設定合適的labels數量


# 將資料拆分為訓練集和測試集
train_inputs, val_inputs, train_instructions, val_instructions = train_test_split(inputs, instructions, test_size=0.1)

# tokenization
train_encodings = tokenizer(train_inputs, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_inputs, truncation=True, padding=True, max_length=128)

# 將 'instruction' 轉換為數字標籤
# 這裡假設 'instruction' 是離散的，並且有一個標籤列表
# instruction_list = list(set(instructions))  # 獲取所有唯一的 instruction
instruction_to_label = {instruction: i for i, instruction in enumerate(instruction_list)}

train_labels = [instruction_to_label[i] for i in train_instructions]
val_labels = [instruction_to_label[i] for i in val_instructions]

# 將資料轉換為 PyTorch Dataset 格式
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = CustomDataset(train_encodings, train_labels)
val_dataset = CustomDataset(val_encodings, val_labels)

# 設置 Trainer
training_args = TrainingArguments(
    output_dir='./results',          # 儲存模型的資料夾
    num_train_epochs=10,              # 訓練的輪數
    per_device_train_batch_size=16,  # 每個裝置的訓練批次大小
    per_device_eval_batch_size=64,   # 每個裝置的評估批次大小
    warmup_steps=500,                # 預熱步數
    weight_decay=0.01,               # 權重衰減
    logging_dir='./logs',            # 訓練日誌資料夾
    evaluation_strategy="epoch",     # 每個epoch結束時進行評估
)

trainer = Trainer(
    model=model,                         # 預訓練模型
    args=training_args,                  # 訓練參數
    train_dataset=train_dataset,         # 訓練資料集
    eval_dataset=val_dataset,            # 評估資料集
)

# 開始訓練
trainer.train()


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the check

Epoch,Training Loss,Validation Loss
1,No log,2.059527
2,No log,2.001364
3,No log,2.008712
4,No log,1.962442
5,No log,1.904433
6,No log,1.922716
7,No log,1.890266
8,No log,1.844484
9,No log,1.862513
10,No log,1.794755


TrainOutput(global_step=250, training_loss=1.7870003662109375, metrics={'train_runtime': 18.1277, 'train_samples_per_second': 219.002, 'train_steps_per_second': 13.791, 'total_flos': 55086705776160.0, 'train_loss': 1.7870003662109375, 'epoch': 10.0})

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

### 單個樣本預測

In [10]:
sample_input = "中醫如何處理肩膀疼痛？"

# 編碼
encoded_input = tokenizer(sample_input, padding=True, truncation=True, return_tensors="pt", max_length=128)
encoded_input = {key: value.to(device) for key, value in encoded_input.items()}

In [11]:

# 模型預測
model.eval()  # 設置為評估模式
with torch.no_grad():
    outputs = model(**encoded_input)

# 獲取預測結果（例如：logits）
logits = outputs.logits
predicted_label = torch.argmax(logits, dim=1).item()  # 預測的類別

# 將預測標籤轉換回原始的指令
predicted_instruction = instruction_list[predicted_label]
print(f"預測的指令是：{predicted_instruction}")


預測的指令是：治療方法


### 其他樣本預測

In [15]:

sample_input_list = ["中醫治療哪些病最有效？", "長期吃中藥好嗎？", "什麼病可以看中醫？", "中醫治療有哪些？", "中醫要持續看嗎？", "皮膚病看中醫有效嗎？", "中醫要看幾次？", "看中醫要看多久？", "同一天可以看西醫跟中醫嗎？", "中醫最多開幾天藥？", "吃中藥隔多久才能吃西藥？", "看中醫調身體有用嗎？"]

for item in sample_input_list:
    # 編碼
    encoded_input = tokenizer(item, padding=True, truncation=True, return_tensors="pt", max_length=128)
    encoded_input = {key: value.to(device) for key, value in encoded_input.items()}
    
    with torch.no_grad():
        outputs = model(**encoded_input)

    # 獲取預測結果（例如：logits）
    logits = outputs.logits
    predicted_label = torch.argmax(logits, dim=1).item()  # 預測的類別

    # 將預測標籤轉換回原始的指令
    predicted_instruction = instruction_list[predicted_label]
    print(f"用戶的問題是:{item} \t 預測的指令是：{predicted_instruction}")

用戶的問題是:中醫治療哪些病最有效？ 	 預測的指令是：治療方法
用戶的問題是:長期吃中藥好嗎？ 	 預測的指令是：其他
用戶的問題是:什麼病可以看中醫？ 	 預測的指令是：治療方法
用戶的問題是:中醫治療有哪些？ 	 預測的指令是：治療方法
用戶的問題是:中醫要持續看嗎？ 	 預測的指令是：治療方法
用戶的問題是:皮膚病看中醫有效嗎？ 	 預測的指令是：治療方法
用戶的問題是:中醫要看幾次？ 	 預測的指令是：治療方法
用戶的問題是:看中醫要看多久？ 	 預測的指令是：其他
用戶的問題是:同一天可以看西醫跟中醫嗎？ 	 預測的指令是：其他
用戶的問題是:中醫最多開幾天藥？ 	 預測的指令是：治療方法
用戶的問題是:吃中藥隔多久才能吃西藥？ 	 預測的指令是：治療方法
用戶的問題是:看中醫調身體有用嗎？ 	 預測的指令是：治療方法
